In [1]:
import os
import re
import json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, BaseOutputParser

# Load environment variables (such as GOOGLE_API_KEY)
load_dotenv()

True

In [2]:
# STEP 1: Define the Blueprints (Schemas)

class JobAnalysisResult(BaseModel):
    """The target blueprint for the Job Analyzer component."""
    job_title: str = Field(description="The formal or official title of the job position.")
    experience_level: str = Field(description="Years or depth of experience requested (e.g., Entry, Mid, 5+ years).")
    mandatory_skills: list[str] = Field(description="Core technical skills or programming languages required.")
    preferred_skills: list[str] = Field(description="Nice-to-have skills, certifications, or bonus tools.")


class JobMatchResult(BaseModel):
    """The target blueprint for the Resume Matcher component."""
    match_percentage: int = Field(description="Match score out of 100 based on core requirements.")
    matching_skills: list[str] = Field(description="Skills present in both the resume and the job description.")
    missing_skills: list[str] = Field(description="Essential skills requested in the job description but missing from the resume.")
    recommendations: list[str] = Field(description="Actionable bullet points on how the candidate can close gaps or improve.")

In [3]:
# STEP 2: Custom Text Guard (Robust JSON Parser)

class RobustJsonParser(BaseOutputParser[dict]):
    """Cleans up the AI's response text and strips away conversational fluff."""
    def parse(self, text: str) -> dict:
        # Search for the outermost JSON curly brackets {}
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if not match:
            raise ValueError(f"Could not find valid JSON boundaries in AI response: {text}")
            
        clean_json_str = match.group(0)
        try:
            return json.loads(clean_json_str)
        except json.JSONDecodeError as e:
            raise ValueError(f"Extracted string is not valid JSON. Error: {e}")

In [4]:
# STEP 3: Centralized AI Factory Configuration

def get_configured_llm(temperature: float = 0.0):
    """
    Function 1: Safely handles initialization of the Gemini model.
    Using a centralized factory ensures we only check the API key in one place.
    """
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("Missing GOOGLE_API_KEY in your environment variables.")
        
    return ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=temperature,  # 0.0 forces strict layout rules; higher allows creative writing
        api_key=api_key
    )

In [5]:
# STEP 4: Job Analyzer Component Functions

def build_analyzer_chain():
    """
    Function 2: Assembles components to extract info from messy job posts.
    """
    # Create formatting instructions for the AI
    standard_parser = JsonOutputParser(pydantic_object=JobAnalysisResult)
    
    # Grab an LLM instance from our factory
    llm = get_configured_llm(temperature=0.1)
    
    # Define instructions
    prompt = ChatPromptTemplate.from_messages([
        (
            "system", 
            "You are an HR data parsing assistant. Extract key information from raw descriptions. "
            "You must return ONLY a raw JSON structure matching the format rules. No conversational fluff.\n{format_instructions}"
        ),
        ("human", "Analyze this raw Job Description text:\n\n{job_description}")
    ])
    
    ready_prompt = prompt.partial(format_instructions=standard_parser.get_format_instructions())
    
    # We chain using the standard RobustJsonParser to handle any formatting errors
    return ready_prompt | llm | RobustJsonParser()

In [6]:
# STEP 5: Job Matcher Component Functions

def build_matcher_chain():
    """
    Function 3: Assembles components to evaluate resumes against structured profiles.
    """
    # Create formatting instructions for the AI
    standard_parser = JsonOutputParser(pydantic_object=JobMatchResult)
    
    # Grab an LLM instance from our factory
    llm = get_configured_llm(temperature=0.0)
    
    # Define scoring instructions
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are an expert technical recruiter. Critically analyze resumes against structured "
            "job elements and provide an objective evaluation. Return ONLY a raw JSON structure.\n{format_instructions}"
        ),
        (
            "human",
            "Evaluate match potential:\n\n"
            "### CANDIDATE RESUME:\n{resume}\n\n"
            "### CLEANED JOB PARAMETERS:\n{job_description}"
        )
    ])
    
    ready_prompt = prompt.partial(format_instructions=standard_parser.get_format_instructions())
    return ready_prompt | llm | RobustJsonParser()

In [7]:
# STEP 6: Coordination and Execution Functions

def run_recruiter_pipeline(resume_text: str, raw_jd_text: str):
    """
    Function 4: The master director. It manages the flow of data between
    the analyzer chain and the matcher chain.
    """
    print(" Stage 1: Building and executing the Job Analyzer...")
    analyzer_chain = build_analyzer_chain()
    # Step A: Parse raw job description text into clean structured dictionary data
    structured_jd = analyzer_chain.invoke({"job_description": raw_jd_text})
    
    print(f"   Identified Target Position : {structured_jd.get('job_title')}")
    print(f"   Experience Level Targeted  : {structured_jd.get('experience_level')}")
    
    print("\n Stage 2: Packaging requirements and running Matcher Engine...")
    matcher_chain = build_matcher_chain()
    
    # Step B: Transform the analyzer's clean dictionary data into a text string so the prompt can digest it
    stringified_requirements = json.dumps(structured_jd)
    
    # Step C: Compare candidate resume against the analyzer's output string
    match_analysis = matcher_chain.invoke({
        "resume": resume_text,
        "job_description": stringified_requirements
    })
    
    return structured_jd, match_analysis


def print_dashboard_report(analysis: dict, match: dict):
    """
    Function 5: Responsible purely for printing a clean UI data dashboard.
    """
    print("\n" + "="*20 + " FINAL RECRUITER INSIGHT REPORT " + "="*20)
    print(f" Evaluated Position  : {analysis.get('job_title', 'N/A')}")
    print(f" Candidate Fit Score : {match.get('match_percentage', 0)}%")
    
    print(f"\n Overlapping Skills  : {', '.join(match.get('matching_skills', []))}")
    print(f" Missing Skill Gaps  : {', '.join(match.get('missing_skills', []))}")
    
    print("\n Actionable Resume Optimization Tips:")
    for tip in match.get("recommendations", []):
        print(f"  • {tip}")
    print("="*72 + "\n")

In [8]:
# STEP 7: Run Application Pipeline

if __name__ == "__main__":
    
    # Simple unformatted sample text parameters
    test_resume = """
    Yash Sharma
    Data Engineer with 3 years of experience.
    Skills: Python, SQL, Git, Apache Spark, Docker.
    Built core backend analytical platforms and automated processing pipelines.
    """

    test_messy_job_description = """
    Hello! We are looking for an experienced Senior AI & Data Developer to join our growing product team.
    Our ideal candidate should have 5+ years of enterprise industry background experience.
    You will primarily write backend architectures using Python and the FastAPI web framework.
    You will also deal with database architecture utilizing core SQL as well as MongoDB.
    Knowledge of deploying code bases within AWS environments is a major bonus!
    """

    try:
        # 1. Run the entire analytical workflow
        clean_jd_data, final_match_data = run_recruiter_pipeline(test_resume, test_messy_job_description)
        
        # 2. Present dashboard metrics cleanly
        print_dashboard_report(clean_jd_data, final_match_data)
        
    except Exception as e:
        print(f"\n Operational Pipeline Error encountered: {e}")

 Stage 1: Building and executing the Job Analyzer...
   Identified Target Position : Senior AI & Data Developer
   Experience Level Targeted  : 5+ years

 Stage 2: Packaging requirements and running Matcher Engine...

==================== FINAL RECRUITER INSIGHT REPORT ====================
 Evaluated Position  : Senior AI & Data Developer
 Candidate Fit Score : 35%

 Overlapping Skills  : Python, SQL
 Missing Skill Gaps  : FastAPI, MongoDB

 Actionable Resume Optimization Tips:
  • Gain at least 2 more years of experience to meet the '5+ years' requirement for a Senior role.
  • Acquire hands-on experience with FastAPI, as it is a mandatory skill for this position.
  • Develop proficiency in MongoDB, which is a critical mandatory skill.
  • Consider learning AWS to align with the preferred skills for the role.
  • If applicable, highlight any experience with AI/ML concepts or projects to better match the 'AI' aspect of the job title.

